# Species-level OOF AUC Report

0.910スコアノートブックと同じOOF評価を行い、種ごとのAUCをCSVで出力する。

### Input
- `birdclef-2026`
- `jaejohn/perch-meta`
- `google/perch_v2_cpu`
- `kdmitrie/bc26-tensorflow-2-20-0`

In [ ]:
import subprocess, sys
from pathlib import Path

_WHL = Path('/kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel')
if not _WHL.exists():
    raise RuntimeError('Add kdmitrie/bc26-tensorflow-2-20-0 as a Notebook input.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    str(_WHL / 'tensorboard-2.20.0-py3-none-any.whl')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    str(_WHL / 'tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl')], check=True)
print('TF installed.')

In [ ]:
import gc, os, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['CUDA_VISIBLE_DEVICES'] = ''
tf.experimental.numpy.experimental_enable_numpy_behavior()
print('TF:', tf.__version__)

In [ ]:
BASE      = Path('/kaggle/input/competitions/birdclef-2026') if Path('/kaggle/input/competitions/birdclef-2026').exists() else Path('/kaggle/input/birdclef-2026')
MODEL_DIR = Path('/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1')

_CACHE_CANDIDATES = [
    Path('/kaggle/input/datasets/jaejohn/perch-meta'),
    Path('/kaggle/working/cache'),
]
CACHE_DIR    = next((d for d in _CACHE_CANDIDATES if (d/'full_perch_meta.parquet').exists() and (d/'full_perch_arrays.npz').exists()), None)
CACHE_EXISTS = CACHE_DIR is not None
WORK_CACHE   = Path('/kaggle/working/cache')
WORK_CACHE.mkdir(exist_ok=True)

SR=32000; WINDOW_SEC=5; WINDOW_SAMPLES=SR*WINDOW_SEC; FILE_SAMPLES=60*SR; N_WINDOWS=12; BATCH_FILES=16
PROBE_PCA_DIM=32; PROBE_MIN_POS=8; PROBE_C=0.25; PROBE_ALPHA=0.40
LAMBDA_EVENT=0.4; LAMBDA_TEXTURE=1.0; LAMBDA_PROXY_TEXTURE=0.8; SMOOTH_TEXTURE_ALPHA=0.35

print('BASE:', BASE)
print('CACHE_EXISTS:', CACHE_EXISTS)

In [ ]:
taxonomy       = pd.read_csv(BASE / 'taxonomy.csv')
soundscape_raw = pd.read_csv(BASE / 'train_soundscapes_labels.csv')
sample_sub     = pd.read_csv(BASE / 'sample_submission.csv')

soundscape_lbls = soundscape_raw.drop_duplicates().reset_index(drop=True)
PRIMARY_LABELS  = sample_sub.columns[1:].tolist()
N_CLASSES       = len(PRIMARY_LABELS)
label_to_idx    = {c: i for i, c in enumerate(PRIMARY_LABELS)}

FNAME_RE = re.compile(r'BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg')

def parse_labels(x):
    if pd.isna(x): return []
    return [t.strip() for t in str(x).split(';') if t.strip()]

def union_labels(series):
    return sorted(set(lbl for x in series for lbl in parse_labels(x)))

def parse_soundscape_filename(name):
    m = FNAME_RE.match(name)
    if not m: return {'site': None, 'hour_utc': -1}
    _, site, _, hms = m.groups()
    return {'site': site, 'hour_utc': int(hms[:2])}

sc_clean = (
    soundscape_lbls.groupby(['filename','start','end'])['primary_label']
    .apply(union_labels).reset_index(name='label_list')
)
sc_clean['end_sec'] = pd.to_timedelta(sc_clean['end']).dt.total_seconds().astype(int)
sc_clean['row_id']  = sc_clean['filename'].str.replace('.ogg','',regex=False) + '_' + sc_clean['end_sec'].astype(str)
meta_cols = sc_clean['filename'].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean  = pd.concat([sc_clean, meta_cols], axis=1)

wpf        = sc_clean.groupby('filename').size()
full_files = sorted(wpf[wpf == N_WINDOWS].index.tolist())
sc_clean['file_fully_labeled'] = sc_clean['filename'].isin(full_files)

Y_SC = np.zeros((len(sc_clean), N_CLASSES), dtype=np.uint8)
for i, labels in enumerate(sc_clean['label_list']):
    for lbl in labels:
        if lbl in label_to_idx: Y_SC[i, label_to_idx[lbl]] = 1

full_truth = sc_clean[sc_clean['file_fully_labeled']].sort_values(['filename','end_sec']).reset_index(drop=False)
print(f'Fully-labeled files: {len(full_files)}, windows: {len(full_truth)}')

In [ ]:
print('Loading Perch...')
birdclassifier = tf.saved_model.load(str(MODEL_DIR))
infer_fn = birdclassifier.signatures['serving_default']
print('Perch loaded.')

bc_labels = pd.read_csv(MODEL_DIR/'assets'/'labels.csv').reset_index().rename(columns={'index':'bc_index','inat2024_fsd50k':'scientific_name'})
NO_LABEL_INDEX = len(bc_labels)

taxonomy_ = taxonomy.copy()
taxonomy_['scientific_name'] = taxonomy_['scientific_name'].astype(str)
mapping = taxonomy_.merge(bc_labels[['scientific_name','bc_index']], on='scientific_name', how='left')
mapping['bc_index'] = mapping['bc_index'].fillna(NO_LABEL_INDEX).astype(int)

label_to_bc   = mapping.set_index('primary_label')['bc_index']
BC_INDICES    = np.array([int(label_to_bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK   = BC_INDICES != NO_LABEL_INDEX
MAPPED_POS    = np.where(MAPPED_MASK)[0].astype(np.int32)
UNMAPPED_POS  = np.where(~MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_INDICES = BC_INDICES[MAPPED_MASK].astype(np.int32)

CLASS_NAME_MAP = taxonomy_.set_index('primary_label')['class_name'].to_dict()
TEXTURE_TAXA   = {'Amphibia','Insecta'}
ACTIVE_CLASSES = [PRIMARY_LABELS[i] for i in np.where(Y_SC.sum(axis=0)>0)[0]]

idx_active_texture = np.array([label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) in TEXTURE_TAXA], dtype=np.int32)
idx_active_event   = np.array([label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) not in TEXTURE_TAXA], dtype=np.int32)
idx_mapped_active_texture   = idx_active_texture[MAPPED_MASK[idx_active_texture]]
idx_mapped_active_event     = idx_active_event[MAPPED_MASK[idx_active_event]]
idx_unmapped_active_texture = idx_active_texture[~MAPPED_MASK[idx_active_texture]]
idx_unmapped_active_event   = idx_active_event[~MAPPED_MASK[idx_active_event]]
idx_unmapped_inactive = np.array([i for i in UNMAPPED_POS if PRIMARY_LABELS[i] not in ACTIVE_CLASSES], dtype=np.int32)

unmapped_df = mapping[mapping['bc_index']==NO_LABEL_INDEX].copy()
proxy_map = {}
for _, row in unmapped_df[~unmapped_df['primary_label'].astype(str).str.contains('son',na=False)].iterrows():
    genus = str(row['scientific_name']).split()[0]
    hits  = bc_labels[bc_labels['scientific_name'].str.match(rf'^{re.escape(genus)}\s', na=False)]
    if len(hits)>0: proxy_map[str(row['primary_label'])] = hits['bc_index'].astype(int).tolist()

SELECTED_PROXY_TARGETS  = sorted([t for t in proxy_map if CLASS_NAME_MAP.get(t)=='Amphibia'])
selected_proxy_pos      = np.array([label_to_idx[c] for c in SELECTED_PROXY_TARGETS], dtype=np.int32)
selected_proxy_pos_to_bc = {label_to_idx[t]: np.array(proxy_map[t], dtype=np.int32) for t in SELECTED_PROXY_TARGETS}
idx_selected_proxy_active_texture     = np.intersect1d(selected_proxy_pos, idx_active_texture)
idx_selected_prioronly_active_texture = np.setdiff1d(idx_unmapped_active_texture, selected_proxy_pos)
idx_selected_prioronly_active_event   = np.setdiff1d(idx_unmapped_active_event, selected_proxy_pos)
print('Label mapping done.')

In [ ]:
def read_soundscape_60s(path):
    y, sr = sf.read(path, dtype='float32', always_2d=False)
    if y.ndim==2: y=y.mean(axis=1)
    if len(y)<FILE_SAMPLES: y=np.pad(y,(0,FILE_SAMPLES-len(y)))
    return y[:FILE_SAMPLES]

def infer_perch_batch(paths, verbose=True):
    paths=[Path(p) for p in paths]
    n_files=len(paths); n_rows=n_files*N_WINDOWS
    row_ids=np.empty(n_rows,dtype=object); filenames=np.empty(n_rows,dtype=object)
    sites=np.empty(n_rows,dtype=object); hours=np.empty(n_rows,dtype=np.int16)
    scores=np.zeros((n_rows,N_CLASSES),dtype=np.float32)
    embeddings=np.zeros((n_rows,1536),dtype=np.float32)
    write_row=0
    for start in tqdm(range(0,n_files,BATCH_FILES),disable=not verbose):
        batch=paths[start:start+BATCH_FILES]; bn=len(batch)
        x=np.empty((bn*N_WINDOWS,WINDOW_SAMPLES),dtype=np.float32); bstart=write_row
        for bi,path in enumerate(batch):
            audio=read_soundscape_60s(path)
            x[bi*N_WINDOWS:(bi+1)*N_WINDOWS]=audio.reshape(N_WINDOWS,WINDOW_SAMPLES)
            meta=parse_soundscape_filename(path.name)
            row_ids[write_row:write_row+N_WINDOWS]=[f'{path.stem}_{t}' for t in range(5,65,5)]
            filenames[write_row:write_row+N_WINDOWS]=path.name
            sites[write_row:write_row+N_WINDOWS]=meta['site']
            hours[write_row:write_row+N_WINDOWS]=meta['hour_utc']
            write_row+=N_WINDOWS
        out=infer_fn(inputs=tf.convert_to_tensor(x))
        logits=out['label'].numpy().astype(np.float32)
        emb=out['embedding'].numpy().astype(np.float32)
        scores[bstart:write_row,MAPPED_POS]=logits[:write_row-bstart,MAPPED_BC_INDICES]
        embeddings[bstart:write_row]=emb
        for pos,bc_idx_arr in selected_proxy_pos_to_bc.items():
            scores[bstart:write_row,pos]=logits[:write_row-bstart,bc_idx_arr].max(axis=1)
        del x,out,logits,emb; gc.collect()
    return pd.DataFrame({'row_id':row_ids,'filename':filenames,'site':sites,'hour_utc':hours}), scores, embeddings

if CACHE_EXISTS:
    print(f'Loading cache from {CACHE_DIR}')
    meta_full=pd.read_parquet(CACHE_DIR/'full_perch_meta.parquet')
    arr=np.load(CACHE_DIR/'full_perch_arrays.npz')
    scores_full_raw=arr['scores_full_raw'].astype(np.float32)
    emb_full=arr['emb_full'].astype(np.float32)
else:
    print('Running Perch on training soundscapes...')
    full_paths=[BASE/'train_soundscapes'/fn for fn in full_files]
    meta_full,scores_full_raw,emb_full=infer_perch_batch(full_paths)
    meta_full.to_parquet(WORK_CACHE/'full_perch_meta.parquet',index=False)
    np.savez_compressed(WORK_CACHE/'full_perch_arrays.npz',scores_full_raw=scores_full_raw,emb_full=emb_full)

full_truth_aligned=full_truth.set_index('row_id').loc[meta_full['row_id']].reset_index(drop=False)
Y_FULL=Y_SC[full_truth_aligned['index'].to_numpy()]
print(f'scores_full_raw: {scores_full_raw.shape}, Y_FULL: {Y_FULL.shape}')

In [ ]:
def fit_prior_tables(prior_df, Y_prior):
    prior_df=prior_df.reset_index(drop=True)
    global_p=Y_prior.mean(axis=0).astype(np.float32)
    site_keys=sorted(prior_df['site'].dropna().astype(str).unique())
    hour_keys=sorted(prior_df['hour_utc'].dropna().astype(int).unique())
    site_to_i,site_n,site_p={},[], []
    for s in site_keys:
        mask=prior_df['site'].astype(str).values==s
        site_to_i[s]=len(site_n); site_n.append(mask.sum()); site_p.append(Y_prior[mask].mean(axis=0))
    site_n=np.array(site_n,dtype=np.float32)
    site_p=np.stack(site_p).astype(np.float32) if site_p else np.zeros((0,Y_prior.shape[1]),np.float32)
    hour_to_i,hour_n,hour_p={},[], []
    for h in hour_keys:
        mask=prior_df['hour_utc'].astype(int).values==h
        hour_to_i[h]=len(hour_n); hour_n.append(mask.sum()); hour_p.append(Y_prior[mask].mean(axis=0))
    hour_n=np.array(hour_n,dtype=np.float32)
    hour_p=np.stack(hour_p).astype(np.float32) if hour_p else np.zeros((0,Y_prior.shape[1]),np.float32)
    sh_to_i,sh_n_list,sh_p_list={},[], []
    for (s,h),idx in prior_df.groupby(['site','hour_utc']).groups.items():
        sh_to_i[(str(s),int(h))]=len(sh_n_list); idx=np.array(list(idx))
        sh_n_list.append(len(idx)); sh_p_list.append(Y_prior[idx].mean(axis=0))
    sh_n=np.array(sh_n_list,dtype=np.float32)
    sh_p=np.stack(sh_p_list).astype(np.float32) if sh_p_list else np.zeros((0,Y_prior.shape[1]),np.float32)
    return dict(global_p=global_p,site_to_i=site_to_i,site_n=site_n,site_p=site_p,
                hour_to_i=hour_to_i,hour_n=hour_n,hour_p=hour_p,sh_to_i=sh_to_i,sh_n=sh_n,sh_p=sh_p)

def prior_logits(sites, hours, tables, eps=1e-4):
    n=len(sites)
    p=np.repeat(tables['global_p'][None,:],n,axis=0).astype(np.float32,copy=True)
    si=np.fromiter((tables['site_to_i'].get(str(s),-1) for s in sites),np.int32,n)
    hi=np.fromiter((tables['hour_to_i'].get(int(h),-1) if int(h)>=0 else -1 for h in hours),np.int32,n)
    shi=np.fromiter((tables['sh_to_i'].get((str(s),int(h)),-1) if int(h)>=0 else -1 for s,h in zip(sites,hours)),np.int32,n)
    for mask_arr,n_arr,p_arr,shrink in [(hi,tables['hour_n'],tables['hour_p'],8.0),(si,tables['site_n'],tables['site_p'],8.0),(shi,tables['sh_n'],tables['sh_p'],4.0)]:
        valid=mask_arr>=0
        if valid.any():
            nv=n_arr[mask_arr[valid]][:,None]
            p[valid]=nv/(nv+shrink)*p_arr[mask_arr[valid]]+(1-nv/(nv+shrink))*p[valid]
    np.clip(p,eps,1-eps,out=p)
    return (np.log(p)-np.log1p(-p)).astype(np.float32)

def smooth_cols(scores,cols,alpha=0.35):
    if alpha<=0 or len(cols)==0: return scores.copy()
    s=scores.copy(); view=s.reshape(-1,N_WINDOWS,s.shape[1]); x=view[:,:,cols]
    view[:,:,cols]=(1-alpha)*x+0.5*alpha*(np.concatenate([x[:,:1,:],x[:,:-1,:]],axis=1)+np.concatenate([x[:,1:,:],x[:,-1:,:]],axis=1))
    return s

def fuse_scores(base,sites,hours,tables):
    scores=base.copy(); prior=prior_logits(sites,hours,tables)
    if len(idx_mapped_active_event): scores[:,idx_mapped_active_event]+=LAMBDA_EVENT*prior[:,idx_mapped_active_event]
    if len(idx_mapped_active_texture): scores[:,idx_mapped_active_texture]+=LAMBDA_TEXTURE*prior[:,idx_mapped_active_texture]
    if len(idx_selected_proxy_active_texture): scores[:,idx_selected_proxy_active_texture]+=LAMBDA_PROXY_TEXTURE*prior[:,idx_selected_proxy_active_texture]
    if len(idx_selected_prioronly_active_event): scores[:,idx_selected_prioronly_active_event]=LAMBDA_EVENT*prior[:,idx_selected_prioronly_active_event]
    if len(idx_selected_prioronly_active_texture): scores[:,idx_selected_prioronly_active_texture]=LAMBDA_TEXTURE*prior[:,idx_selected_prioronly_active_texture]
    if len(idx_unmapped_inactive): scores[:,idx_unmapped_inactive]=-8.0
    return smooth_cols(scores,idx_active_texture,alpha=SMOOTH_TEXTURE_ALPHA).astype(np.float32), prior

print('Functions defined.')

In [ ]:
# OOF評価
gkf=GroupKFold(n_splits=5); groups=meta_full['site'].to_numpy()
oof_base=np.zeros_like(scores_full_raw,dtype=np.float32)
oof_prior=np.zeros_like(scores_full_raw,dtype=np.float32)

for _,va_idx in tqdm(list(gkf.split(scores_full_raw,groups=groups)),desc='OOF'):
    va_idx=np.sort(va_idx)
    val_sites=set(meta_full.iloc[va_idx]['site'].tolist())
    prior_m=~sc_clean['site'].isin(val_sites).values
    tables=fit_prior_tables(sc_clean.loc[prior_m].reset_index(drop=True),Y_SC[prior_m])
    oof_base[va_idx],oof_prior[va_idx]=fuse_scores(scores_full_raw[va_idx],meta_full.iloc[va_idx]['site'].to_numpy(),meta_full.iloc[va_idx]['hour_utc'].to_numpy(),tables)

# Probe学習・OOF予測
emb_scaler=StandardScaler(); emb_scaled=emb_scaler.fit_transform(emb_full)
n_comp=min(PROBE_PCA_DIM,emb_scaled.shape[0]-1,emb_scaled.shape[1])
emb_pca=PCA(n_components=n_comp); Z_FULL=emb_pca.fit_transform(emb_scaled).astype(np.float32)

def seq_features_1d(v):
    x=v.reshape(-1,N_WINDOWS)
    prev=np.concatenate([x[:,:1],x[:,:-1]],axis=1).reshape(-1)
    nxt=np.concatenate([x[:,1:],x[:,-1:]],axis=1).reshape(-1)
    return prev,nxt,np.repeat(x.mean(1),N_WINDOWS),np.repeat(x.max(1),N_WINDOWS)

def build_class_features(Z,raw_col,prior_col,base_col):
    p,n,m,mx=seq_features_1d(base_col)
    return np.concatenate([Z,raw_col[:,None],prior_col[:,None],base_col[:,None],p[:,None],n[:,None],m[:,None],mx[:,None]],axis=1).astype(np.float32)

oof_final=oof_base.copy()
for _,va_idx in tqdm(list(gkf.split(scores_full_raw,groups=groups)),desc='Probe OOF'):
    tr_idx=np.setdiff1d(np.arange(len(scores_full_raw)),va_idx)
    pos_cnt=Y_FULL[tr_idx].sum(axis=0)
    for ci in np.where(pos_cnt>=PROBE_MIN_POS)[0]:
        y_tr=Y_FULL[tr_idx,ci]
        if y_tr.sum()==0 or y_tr.sum()==len(y_tr): continue
        X_tr=build_class_features(Z_FULL[tr_idx],scores_full_raw[tr_idx,ci],oof_prior[tr_idx,ci],oof_base[tr_idx,ci])
        X_va=build_class_features(Z_FULL[va_idx],scores_full_raw[va_idx,ci],oof_prior[va_idx,ci],oof_base[va_idx,ci])
        clf=LogisticRegression(C=PROBE_C,max_iter=400,solver='liblinear',class_weight='balanced')
        clf.fit(X_tr,y_tr)
        pred=clf.decision_function(X_va).astype(np.float32)
        oof_final[va_idx,ci]=(1-PROBE_ALPHA)*oof_base[va_idx,ci]+PROBE_ALPHA*pred

print('OOF done.')

In [ ]:
# 種ごとのAUCを計算してCSV出力
results = []
for i, label in enumerate(PRIMARY_LABELS):
    n_pos = int(Y_FULL[:, i].sum())
    if n_pos == 0:
        auc_base  = None
        auc_final = None
    else:
        auc_base  = float(roc_auc_score(Y_FULL[:, i], oof_base[:, i]))
        auc_final = float(roc_auc_score(Y_FULL[:, i], oof_final[:, i]))
    results.append({
        'primary_label': label,
        'n_positive':    n_pos,
        'auc_base':      auc_base,
        'auc_final':     auc_final,
        'perch_mapped':  bool(MAPPED_MASK[i]),
    })

result_df = pd.DataFrame(results)
result_df = result_df.merge(
    taxonomy[['primary_label','common_name','scientific_name','class_name']],
    on='primary_label', how='left'
)
result_df = result_df[['primary_label','common_name','scientific_name','class_name',
                        'n_positive','perch_mapped','auc_base','auc_final']]

# 全体AUC
keep = result_df['n_positive'] > 0
overall_base  = roc_auc_score(Y_FULL[:, keep.values], oof_base[:, keep.values],  average='macro')
overall_final = roc_auc_score(Y_FULL[:, keep.values], oof_final[:, keep.values], average='macro')
print(f'Overall OOF AUC (base)  : {overall_base:.4f}')
print(f'Overall OOF AUC (final) : {overall_final:.4f}')

result_df.to_csv('/kaggle/working/species_auc.csv', index=False)
print(f'Saved: species_auc.csv  ({len(result_df)} rows)')
result_df.sort_values('auc_final', ascending=False).head(10)